## 0. Príprava prostredia

Tracking aj Registry ukladáme do lokálnej **SQLite** databázy (`mlflow.db`) — odporúčaný backend, ktorý podporuje aj Model Registry. Artefakty (modely, grafy) idú do `./mlartifacts`.

In [103]:
# %pip install torch

In [104]:


import mlflow
import sklearn
import pandas as pd

print("MLflow:      ", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)

# Centralny tracking + registry backend (lokalna SQLite databaza)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
print("Tracking URI:", mlflow.get_tracking_uri())


MLflow:       3.14.0
scikit-learn: 1.9.0
Tracking URI: sqlite:///mlflow.db


### Dataset — načítanie a rozdelenie

Diabetes dataset: 10 číselných príznakov (vek, BMI, krvný tlak, krvné hodnoty), cieľ = miera progresie ochorenia o rok neskôr (regresia).

In [105]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)
df = data.frame

X = df.drop(columns=["target"])
y = df["target"]

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=0
)

print("Shape:", df.shape, "| train:", X_train.shape[0], "| test:", X_test.shape[0])
df.head()

Shape: (442, 11) | train: 309 | test: 133


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


---
# MLflow Tracking

**Tracking** je jadro MLflow. Každý **beh (run)** zaloguje vstupné **parametre**, výstupné **metriky** a **artefakty** (modely, grafy, kód), plus ľubovoľné **tagy**. Behy patria do **experimentov**.

```
experiment ─┬─ run ── params · metrics · artifacts · tags
            ├─ run ── ...
            └─ run ── ...
```

### Čo si v tejto kapitole vyskúšaš

| Sekcia | Možnosť MLflow | Stručne |
|:------:|----------------|---------|
| **1.1** | Experimenty | vytvor / zvoľ / otaguj / zmaž experiment (module aj `MlflowClient`) |
| **1.2** | Logovanie behu | `log_param(s)`, `log_metric(s)`, `log_artifact`, `log_figure` |
| **1.3** | Autologging | `mlflow.autolog()` — automaticky params, metriky, model aj signature |
| **1.4** | Krivka učenia | strata po epochách cez `log_metric(..., step=)` — PyTorch NN |
| **1.5** | Tagy | `set_tag(s)`, čítanie, úprava/mazanie, filtrovanie podľa tagu |
| **1.6** | Viac behov | vnorené (parent-child) behy — sweep hyperparametrov ElasticNet |
| **1.7** | Dáta o behu | `last_active_run()` a `get_run(run_id)` |
| **1.8** | Querying | `search_runs()` s `filter_string`, `order_by`, `max_results` |

> Všetko sa loguje do lokálnej SQLite (`sqlite:///mlflow.db`). Na konci kapitoly si výsledky pozrieš vo webovom **MLflow UI**.

## 1.1 Práca s experimentami (MLflow module vs. MlflowClient)

Experiment = logická skupina behov (runs). Môžeme pracovať 2 spôsobmi, prostredníctvom clienta alebo modulu. Práca s clientom pre localhost je ekvivalentná práci s modulom.

| Operácia | MLflow module | MlflowClient |
|----------|---------------|--------------|
| Vytvoriť | `mlflow.create_experiment("Name")` | `client.create_experiment("Name")` |
| Otagovať | `mlflow.set_experiment_tag(k, v)` | `client.set_experiment_tag(id, k, v)` |
| Zmazať | `mlflow.delete_experiment(id)` | `client.delete_experiment(id)` |
| Zvoliť aktívny | `mlflow.set_experiment("Name")` | — |

In [106]:
exp_name = "diabetes-tracking"
if mlflow.get_experiment_by_name(exp_name) is None:
    mlflow.create_experiment(exp_name)


## 1.2 Spustenie behu a logovanie (`start_run` + logging)

Nový beh = nový tréning modelu, spúšťa sa cez `mlflow.start_run()` (a ukončuje `mlflow.end_run()`, alebo automaticky pri `with` bloku). Logujeme (slidy *Logging to MLflow Tracking*):
- **Metriky:** `log_metric("accuracy", 0.9)` · `log_metrics({...})`
- **Parametre:** `log_param("n_jobs", 1)` · `log_params({...})`
- **Artefakty:** `log_artifact("file.py")` · `log_artifacts("./dir/")`

In [107]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

params = {"alpha": 0.005, "l1_ratio": 0.5, "random_state": 42}

mlflow.set_experiment("diabetes-tracking")  # poistka: nech beh nespadne do Default
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()
# dam si nazov behu na baseline, s nejakymi zakladnymi parametrami
run = mlflow.start_run(run_name="elasticnet-baseline")

# --- trening + predikcia ---
model = ElasticNet(**params)
model.fit(X_train, y_train)
preds = model.predict(X_test)

rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

# --- log parametrov a metrik ---
mlflow.log_params(params)
mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

# --- log kodu (samotny notebook ako artefakt) ---
# mlflow.log_artifact("mlflow_workshop.ipynb", artifact_path="code")

# --- jednoduche vizualizacie (logujeme priamo ako obrazky) ---
# 1) Predikcia vs. skutocnost
fig1, ax1 = plt.subplots(figsize=(5, 5))
ax1.scatter(y_test, preds, alpha=0.6, edgecolor="k")
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
ax1.plot(lims, lims, "r--", lw=1)
ax1.set_xlabel("Skutocnost (y_test)")
ax1.set_ylabel("Predikcia")
ax1.set_title("Predikcia vs. skutocnost")
mlflow.log_figure(fig1, "plots/pred_vs_actual.png")
plt.close(fig1)

# 2) Histogram rezidui
residuals = y_test - preds
fig2, ax2 = plt.subplots(figsize=(5, 4))
ax2.hist(residuals, bins=25, edgecolor="k")
ax2.axvline(0, color="r", linestyle="--", lw=1)
ax2.set_xlabel("Rezidua (y_test - predikcia)")
ax2.set_ylabel("Pocet")
ax2.set_title("Rozdelenie rezidui")
mlflow.log_figure(fig2, "plots/residuals.png")
plt.close(fig2)

# --- vypis ---
print("run_id:", run.info.run_id, "| run_name:", run.info.run_name)
print(f"RMSE={rmse:.2f} | MAE={mae:.2f} | R2={r2:.3f}")
print("Zalogovane: params, metriky, kod (notebook) + 2 grafy.")

mlflow.end_run()


run_id: 57963bfccaac4901a4b335b00a3922b6 | run_name: elasticnet-baseline
RMSE=56.25 | MAE=45.09 | R2=0.380
Zalogovane: params, metriky, kod (notebook) + 2 grafy.


Nižšie môžeme získať posledný aktívny beh a zobraziť si zalogovaný obrázok

In [108]:
from PIL import Image

last_run = mlflow.last_active_run()
run_id = last_run.info.run_id

client = mlflow.tracking.MlflowClient()
local_path = client.download_artifacts(run_id, "plots/pred_vs_actual.png", dst_path="./downloads")

img = Image.open(local_path)
img.show()

In [109]:
experiment = mlflow.get_experiment_by_name("diabetes-tracking")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'elasticnet-baseline'",
    order_by=["start_time DESC"]
)

# run_id = runs.iloc[0]["run_id"]
# print(run_id)
print(runs)

                              run_id experiment_id    status  \
0   57963bfccaac4901a4b335b00a3922b6             1  FINISHED   
1   a8c1b32a473a479bb7bbb1d6389d5bb7             1  FINISHED   
2   83308598256f48199d3fe26732834cd1             1  FINISHED   
3   8551303ab4854463954c9ba1e1e31ec7             1  FINISHED   
4   9ed6d23ee09f44939d9af7a2fea4f0a9             1  FINISHED   
5   9df2e07b459044c78a90f85daacc6068             1  FINISHED   
6   0fdb87556223424983bc6595d11522fe             1  FINISHED   
7   3699eeada7cc47c4b1c308d6e4a24349             1  FINISHED   
8   dae22f5757b748bdbcac083ea355c312             1  FINISHED   
9   04d8a0917d38437790569c25da781e39             1  FINISHED   
10  4c26f474447b4beb96a9cc6953dece0c             1   RUNNING   

                                         artifact_uri  \
0   file:c:/Users/milan/PycharmProjects/PythonProj...   
1   file:c:/Users/milan/PycharmProjects/PythonProj...   
2   file:c:/Users/milan/PycharmProjects/PythonProj...   
3  

## 1.3 Autologging — `mlflow.autolog()`

Namiesto ručného volania `log_params` / `log_metrics` / `log_model` vie MLflow zachytiť všetko **automaticky**. Stačí raz zapnúť autolog pre danú knižnicu a následný `fit()` zaloguje:

- **hyperparametre** modelu (z `get_params()`)
- **tréningové metriky**
- **model** vrátane **podpisu (signature)** a `input_example`
- prípadné grafy / artefakty podporované danou integráciou

`mlflow.autolog()` zapne autolog pre všetky podporované knižnice; `mlflow.sklearn.autolog()` cielene pre scikit-learn. Porovnaj s ručným logovaním v 1.2 — rovnaký výsledok, jeden riadok.

> Autolog necháme zapnutý len pre túto bunku a na konci ho vypneme (`disable=True`), aby ďalšie sekcie logovali tak ako doteraz.

In [110]:
# Zapni autolog pre scikit-learn (pre vsetky podporovane kniznice naraz: mlflow.autolog())
mlflow.sklearn.autolog()

mlflow.set_experiment("diabetes-tracking")
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()

# Ziadne explicitne log_* volania — params, metriky, model aj signature
# zachyti autolog automaticky pri fit().
with mlflow.start_run(run_name="elasticnet-autolog") as run:
    model = ElasticNet(alpha=0.005, l1_ratio=0.5, random_state=42)
    model.fit(X_train, y_train)

# Pozri, co sa zalogovalo bez jedineho log_* volania
auto = mlflow.get_run(run.info.run_id)
print("Autolog beh:", run.info.run_id)
print("Zalogovane parametre:", auto.data.params)
print("Zalogovane metriky:  ", {k: round(v, 3) for k, v in auto.data.metrics.items()})

# Vypni autolog, aby dalsie sekcie logovali rucne (a sweep v 1.6 nelogoval dvojmo)
mlflow.sklearn.autolog(disable=True)

2026/06/30 17:41:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Autolog beh: b616a6aa9cbc46dc8398f6a827c1ea66
Zalogovane parametre: {'alpha': '0.005', 'copy_X': 'True', 'fit_intercept': 'True', 'l1_ratio': '0.5', 'max_iter': '1000', 'positive': 'False', 'precompute': 'False', 'random_state': '42', 'selection': 'cyclic', 'tol': '0.0001', 'warm_start': 'False'}
Zalogovane metriky:   {'training_mean_squared_error': 3282.145, 'training_mean_absolute_error': 48.654, 'training_r2_score': 0.478, 'training_root_mean_squared_error': 57.29, 'training_score': 0.478}


## 1.4 Logovanie straty počas tréningu (PyTorch NN)

Pri iteratívnom tréningu (neurónová sieť) chceme vidieť **vývoj straty po epochách**. `mlflow.log_metric("train_loss", value, step=epoch)` zaloguje metriku ako **časový rad** — v UI sa zobrazí ako **krivka učenia**.

Ten istý beh ukazuje aj ostatné možnosti logovania:
- **parametre:** `log_param` (jeden) · `log_params` (viac naraz)
- **tag:** `set_tag`
- **strata:** `log_metric(..., step=epoch)` → krivka
- **finálne metriky:** `log_metrics`
- **kód + graf:** `log_artifact` · `log_figure`

In [111]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

# --- Standardizacia vstupov (NN potrebuje skalovane priznaky) ---
scaler = StandardScaler().fit(X_train)
Xtr = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
Xte = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
ytr = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)

# --- Maly MLP: 10 -> 32 -> 16 -> 1 ---
net = nn.Sequential(
    nn.Linear(Xtr.shape[1], 32), nn.ReLU(),
    nn.Linear(32, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
nn_params = {"hidden_1": 32, "hidden_2": 16, "lr": 0.01, "epochs": 200, "optimizer": "Adam"}
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=nn_params["lr"])

mlflow.set_experiment("diabetes-tracking")  # poistka: nech beh nespadne do Default
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()
run = mlflow.start_run(run_name="pytorch-mlp")

# (1) PARAMETRE — jeden naraz (log_param) aj viac naraz (log_params)
mlflow.log_param("optimizer", nn_params["optimizer"])
mlflow.log_params({k: nn_params[k] for k in ["hidden_1", "hidden_2", "lr", "epochs"]})

# (2) TAG behu
mlflow.set_tag("model_family", "neural-net")

# --- Trening + LOGOVANIE STRATY po kazdej epoche (metrika so `step`) ---
losses = []
for epoch in range(nn_params["epochs"]):
    optimizer.zero_grad()
    loss = criterion(net(Xtr), ytr)
    loss.backward()
    optimizer.step()
    mlflow.log_metric("train_loss", loss.item(), step=epoch)   # -> krivka ucenia v UI
    losses.append(loss.item())

# --- Evaluacia ---
net.eval()
with torch.no_grad():
    preds = net(Xte).numpy().reshape(-1)
rmse = root_mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

# (3) FINALNE METRIKY (log_metrics — viac naraz)
mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

# (4) KOD ako artefakt (samotny notebook)
mlflow.log_artifact("mlflow_tracking.ipynb", artifact_path="code")

# (5) GRAF — krivka straty ako obrazok
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses)
ax.set_xlabel("Epocha"); ax.set_ylabel("Train loss (MSE)"); ax.set_title("Krivka ucenia")
mlflow.log_figure(fig, "plots/loss_curve.png")
plt.close(fig)

# (6) PREDIKCNE VIZUALIZACIE
# 6a) Scatter: predikcia vs. skutocnost
fig_s, ax_s = plt.subplots(figsize=(5, 5))
ax_s.scatter(y_test, preds, alpha=0.6, edgecolor="k")
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
ax_s.plot(lims, lims, "r--", lw=1, label="idealna zhoda")
ax_s.set_xlabel("Skutocnost (y_test)")
ax_s.set_ylabel("Predikcia")
ax_s.set_title("Scatter: predikcia vs. skutocnost")
ax_s.legend()
mlflow.log_figure(fig_s, "plots/pred_vs_actual.png")
plt.close(fig_s)

# 6b) Heatmapa: 2D hustota (predikcia vs. skutocnost)
fig_h, ax_h = plt.subplots(figsize=(5.5, 4.5))
hb = ax_h.hist2d(y_test, preds, bins=15, cmap="viridis")
fig_h.colorbar(hb[3], ax=ax_h, label="Pocet vzoriek")
ax_h.plot(lims, lims, "w--", lw=1)
ax_h.set_xlabel("Skutocnost (y_test)")
ax_h.set_ylabel("Predikcia")
ax_h.set_title("Heatmapa hustoty predikcii")
mlflow.log_figure(fig_h, "plots/pred_actual_heatmap.png")
plt.close(fig_h)

print("run_id:", run.info.run_id, "| run_name:", run.info.run_name)
print(f"final train_loss={losses[-1]:.1f} | RMSE={rmse:.2f} | MAE={mae:.2f} | R2={r2:.3f}")

mlflow.end_run()


run_id: 83ef3306318942e6845dd77dcff53e79 | run_name: pytorch-mlp
final train_loss=2396.5 | RMSE=57.16 | MAE=45.70 | R2=0.360


In [112]:
plt.show(fig_h)

## 1.5 Tagovanie behov — `set_tag` / `set_tags`

Tagy sú ľubovoľné `key: value` metadáta na behu (autor, dataset, fáza...). Na rozdiel od parametrov ich vieš **meniť aj po skončení behu** a hodia sa najmä na **organizáciu a filtrovanie**.

- **jeden tag:** `mlflow.set_tag("model_family", "linear")`
- **viac naraz:** `mlflow.set_tags({...})`
- **čítanie:** `mlflow.get_run(run_id).data.tags`
- **úprava / mazanie (cez klienta):** `client.set_tag(...)` · `client.delete_tag(...)`
- **filtrovanie:** `search_runs(filter_string="tags.dataset = 'diabetes'")`

> Tagy začínajúce na `mlflow.` sú systémové (napr. `mlflow.runName`).

In [113]:
3
mlflow.set_experiment("diabetes-tracking")

if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()

run = mlflow.start_run(run_name="run-with-tags")   # zachytime run objekt
tagged_run_id = run.info.run_id

# rychly model, nech mame co logovat
m = ElasticNet(alpha=0.005, l1_ratio=0.5, random_state=42).fit(X_train, y_train)
mlflow.log_metric("rmse", root_mean_squared_error(y_test, m.predict(X_test)))

# (1) JEDEN tag naraz (na aktivny beh)
mlflow.set_tag("model_family", "linear")

# (2) VIAC tagov naraz (dict)
mlflow.set_tags({"dataset": "diabetes", "stage": "experiment", "author": "Laco"})

# (3) CITANIE tagov spat (systemove "mlflow.*" odfiltrujeme)
tags = mlflow.get_run(tagged_run_id).data.tags
print("Tagy behu:", {k: v for k, v in tags.items() if not k.startswith("mlflow.")})


# (5) FILTROVANIE behov podla tagu
hits = mlflow.search_runs(
    experiment_names=["diabetes-tracking"],
    filter_string="tags.dataset = 'diabetes'",
)
print("Behov s tagom dataset=diabetes:", len(hits))
print("Po uprave:", {k: v for k, v in mlflow.get_run(tagged_run_id).data.tags.items()
                     if k in ("stage", "author", "dataset")})

mlflow.end_run()

Tagy behu: {'model_family': 'linear', 'dataset': 'diabetes', 'stage': 'experiment', 'author': 'Laco'}
Behov s tagom dataset=diabetes: 8
Po uprave: {'dataset': 'diabetes', 'stage': 'experiment', 'author': 'Laco'}


## 1.6 Viac behov — vnorené (parent-child) behy

Pri ladení hyperparametrov je zvykom zastrešiť celý **sweep** jedným **rodičovským behom** a každú kombináciu logovať ako **vnorený (child) beh** cez `mlflow.start_run(nested=True)`. V UI sa child behy zobrazia zbalené pod rodičom a dajú sa rozbaliť.

- **rodič:** `with mlflow.start_run(run_name="elasticnet-sweep") as parent:`
- **dieťa:** vnútri `with mlflow.start_run(..., nested=True):`
- do rodiča zvykneme zalogovať **súhrn** sweepu (počet behov, najlepšie RMSE)

Tieto behy potom budeme **filtrovať** (1.8).

In [114]:
from itertools import product

mlflow.set_experiment("diabetes-tracking")
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()

# Rodicovsky (parent) beh zastresuje cely sweep;
# kazda kombinacia hyperparametrov bezi ako vnoreny (child) beh.
with mlflow.start_run(run_name="elasticnet-sweep") as parent:
    print("Parent run_id:", parent.info.run_id)

    results = []
    for alpha, l1 in product([0.001, 0.005, 0.01], [0.2, 0.5, 0.8]):
        # nested=True -> beh sa v UI zobrazi vnoreny pod rodicom
        with mlflow.start_run(run_name=f"en-a{alpha}-l{l1}", nested=True):
            p = {"alpha": alpha, "l1_ratio": l1, "random_state": 42}
            m = ElasticNet(**p).fit(X_train, y_train)
            preds = m.predict(X_test)
            rmse = root_mean_squared_error(y_test, preds)
            mlflow.log_params(p)
            mlflow.log_metrics({
                "rmse": rmse,
                "mae": mean_absolute_error(y_test, preds),
                "r2": r2_score(y_test, preds),
            })
            mlflow.sklearn.log_model(m, name="model")
            results.append((rmse, alpha, l1))

    # Do RODICA zalogujeme suhrn celeho sweepu
    best_rmse, best_alpha, best_l1 = min(results)
    mlflow.log_param("sweep_runs", len(results))
    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.set_tags({"best_alpha": best_alpha, "best_l1_ratio": best_l1})

print(f"Spustenych {len(results)} vnorenych behov pod rodicom.")
print(f"Najlepsie RMSE={best_rmse:.2f} (alpha={best_alpha}, l1_ratio={best_l1})")

Parent run_id: 82e60fa452224d6d86252c2c2fe5ed79
Spustenych 9 vnorenych behov pod rodicom.
Najlepsie RMSE=55.01 (alpha=0.001, l1_ratio=0.2)


## 1.7 Dáta o behu — `last_active_run` a `get_run`

Po skončení behu sa k nemu vieme vrátiť: `last_active_run()` vráti **posledný beh aktívny v tomto kerneli**, `get_run(run_id)` načíta **ľubovoľný** beh podľa ID.

In [115]:
# Pre istotu spravime kratky beh, aby last_active_run() mal co vratit
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()

# last_active_run() vrati POSLEDNY beh aktivny v TOMTO kerneli.
run = mlflow.last_active_run()
if run is None:
    print("Ziadny aktivny beh v tomto kerneli. Spusti najprv niektoru logovaciu bunku (1.2 / 1.4 / 1.5).")
else:
    run_id = run.info.run_id
    print("last_active_run id:", run_id, "| run_name:", run.info.run_name)

    # Kompletne data behu podla run_id (funguje pre lubovolny run_id)
    fetched = mlflow.get_run(run_id)
    print("Parametre:", fetched.data.params)
    print("Metriky:  ", {k: round(v, 3) for k, v in fetched.data.metrics.items()})

last_active_run id: 82e60fa452224d6d86252c2c2fe5ed79 | run_name: elasticnet-sweep
Parametre: {'sweep_runs': '9'}
Metriky:   {'best_rmse': 55.011}


## 1.8 Querying runs — `search_runs`

`mlflow.search_runs()` vráti behy ako **pandas DataFrame**. Možnosti filtrovania (slidy *Filtering run searches*):
- **`experiment_names`** — z ktorých experimentov
- **`filter_string`** — SQL-like query, napr. `"metrics.rmse < 56"`
- **`order_by`** — napr. `["metrics.rmse ASC"]`
- **`max_results`** — limit počtu

In [116]:
# Vsetky behy, zoradene podla RMSE (najnizsie = najlepsie)
runs = mlflow.search_runs(
    experiment_names=["diabetes-tracking"],
    order_by=["metrics.rmse ASC"],
)
cols = ["run_id", "tags.mlflow.runName",
        "params.alpha", "params.l1_ratio",
        "metrics.rmse", "metrics.r2"]
runs[cols].head(10)

,run_id,tags.mlflow.runName,params.alpha,params.l1_ratio,metrics.rmse,metrics.r2
0,60c88e2f4af5450aa1a04aad69918ec5,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
1,36d34932d6eb4de9919386cf27fd81ad,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
2,177061fb4299442f85f776235da8c5c4,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
3,e645cd770b2c44f0890962a648b6b521,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
4,b7a348da2f764fbf9e4df0887b981676,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
5,fc332351e24a46fca4b9bd965c2354ba,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
6,efc36453835a493dbe717ef6caf11f9c,en-a0.001-l0.2,0.001,0.2,55.010936,0.406800
7,e9e1a06cf8114050abf6db5b9fdeb114,en-a0.001-l0.5,0.001,0.5,55.070607,0.405513
8,5737792528f34905806f64d76bc0c651,en-a0.001-l0.5,0.001,0.5,55.070607,0.405513
9,fcb2152baced4b12b7e4a9d6bb7bd233,en-a0.001-l0.5,0.001,0.5,55.070607,0.405513


In [117]:
# Priklad filter_string (analogia slidu: "metrics.f1_score > 0.60")
rmse_filter = "metrics.rmse < 56"
good = mlflow.search_runs(
    experiment_names=["diabetes-tracking"],
    filter_string=rmse_filter,
    order_by=["metrics.r2 DESC"],
    max_results=5,
)
print(f"Vyhovujucich behov: {len(good)}")
good[["tags.mlflow.runName", "metrics.rmse", "metrics.r2"]]

Vyhovujucich behov: 5


,tags.mlflow.runName,metrics.rmse,metrics.r2
0,en-a0.001-l0.2,55.010936,0.4068
1,en-a0.001-l0.2,55.010936,0.4068
2,en-a0.001-l0.2,55.010936,0.4068
3,en-a0.001-l0.2,55.010936,0.4068
4,en-a0.001-l0.2,55.010936,0.4068


In [118]:
# Najlepsi beh si zapamatame pre dalsie kapitoly
best = runs.sort_values("metrics.rmse").iloc[0]
best_run_id = best["run_id"]
print("Najlepsi beh:", best["tags.mlflow.runName"], "| run_id:", best_run_id)
print(f"RMSE={best['metrics.rmse']:.2f} | R2={best['metrics.r2']:.3f}")

Najlepsi beh: en-a0.001-l0.2 | run_id: 60c88e2f4af5450aa1a04aad69918ec5
RMSE=55.01 | R2=0.407


---
## Zhrnutie + MLflow UI

V tejto kapitole sme prešli celý **Tracking** tok:
**experimenty** → **logovanie** (params · metriky · artefakty · tagy) → **krivka straty** → **viac behov** → **querying & filtrovanie**.

Výsledky si pozri vo webovom **UI** — v termináli z priečinka projektu spusti:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Otvor **http://localhost:5000**. V UI si vieš prezrieť:

- **Experimenty a behy** — zoznam všetkých behov, ich stav, čas a trvanie.
- **Parametre a metriky** — finálne hodnoty každého behu v prehľadnej tabuľke.
- **Priebeh tréningu** — metriky logované so `step` (napr. `train_loss`) sa zobrazia ako **krivka učenia** v čase.
- **Artefakty** — uložené grafy (scatter, heatmapa, krivka straty), model aj zalogovaný kód, priamo otvoriteľné v prehliadači.
- **Tagy** — filtrovanie a organizácia behov podľa vlastných tagov.
- **Porovnanie behov** — označíš viac behov a porovnáš ich parametre a metriky vedľa seba (vhodné na výber najlepšieho modelu).

## 1.9 Načítanie modelu — `mlflow.sklearn.load_model`

Uložený model načítame späť pomocou jeho **URI**. Formát URI pre beh: `runs:/<run_id>/<artifact_path>`.

| Spôsob | URI |
|--------|-----|
| Z behu | `runs:/<run_id>/model` |
| Z Model Registry | `models:/<name>/<version>` |

`mlflow.pyfunc.load_model()` načíta **akýkoľvek** flavor ako generický PyFunc (`.predict()`). Pre sklearn je k dispozícii aj natívny `mlflow.sklearn.load_model()`, ktorý vráti priamo `sklearn` objekt.

In [119]:
# best_run_id je nastaveny v bunke 1.8 (najlepsi beh podla RMSE)
model_uri = f"runs:/{best_run_id}/model"
print("Nacitavam model z:", model_uri)

# Nacitanie ako sklearn objekt (vrati ElasticNet priamo)
loaded_model = mlflow.sklearn.load_model(model_uri)
print("Typ modelu:", type(loaded_model))
print("Parametre:", loaded_model.get_params())

# Overenie: predikcia nacitanym modelom sa zhoduje s povodnou
preds_loaded = loaded_model.predict(X_test)
rmse_loaded = root_mean_squared_error(y_test, preds_loaded)
print(f"\nRMSE nacitaneho modelu: {rmse_loaded:.2f}")
print("Zhoda s povodnou predikciou:", best["metrics.rmse"].round(2) == round(rmse_loaded, 2))

Nacitavam model z: runs:/60c88e2f4af5450aa1a04aad69918ec5/model
Typ modelu: <class 'sklearn.linear_model._coordinate_descent.ElasticNet'>
Parametre: {'alpha': 0.001, 'copy_X': True, 'fit_intercept': True, 'l1_ratio': 0.2, 'max_iter': 1000, 'positive': False, 'precompute': False, 'random_state': 42, 'selection': 'cyclic', 'tol': 0.0001, 'warm_start': False}

RMSE nacitaneho modelu: 55.01
Zhoda s povodnou predikciou: True


## 1.10 Automatická evaluácia — `mlflow.models.evaluate()`

Pred povýšením modelu ho vieme **automaticky vyhodnotiť** jediným volaním. `mlflow.models.evaluate()` vezme model (cez URI), evaluačný dataset a typ úlohy a do behu zaloguje **sadu metrík** (RMSE, MAE, R², MAPE...) plus **diagnostické artefakty** (grafy, tabuľky).

- **model:** URI, napr. `runs:/<run_id>/model`
- **data:** DataFrame, kde je cieľ samostatný stĺpec
- **targets:** názov cieľového stĺpca
- **model_type:** `"regressor"` (náš prípad) / `"classifier"`

> Od MLflow 3.0 sa pôvodné `mlflow.evaluate` označuje ako deprecated — používame `mlflow.models.evaluate` (rovnaké API, bez varovania).
>
> SHAP explainability vypíname cez `evaluator_config={"log_model_explainability": False}`, takže netreba inštalovať balík `shap`. Základné metriky a grafy sa zalogujú aj tak.

In [120]:
# mlflow.models.evaluate potrebuje dataset, kde je cielovy stlpec sucastou DataFrame
eval_df = X_test.copy()
eval_df["target"] = y_test

mlflow.set_experiment("diabetes-tracking")
if mlflow.active_run():  # poistka: zatvor pripadny otvoreny beh z predoslej bunky
    mlflow.end_run()

# Vyhodnotime najlepsi model (z bunky 1.8) na testovej sade.
# log_model_explainability=False vypne SHAP (netreba balik shap).
with mlflow.start_run(run_name="evaluate-best") as run:
    result = mlflow.models.evaluate(
        f"runs:/{best_run_id}/model",
        data=eval_df,
        targets="target",
        model_type="regressor",
        evaluators="default",
        evaluator_config={"log_model_explainability": False},
    )

print("Evaluacny beh:", run.info.run_id)
print("\nMetriky:")
for k, v in result.metrics.items():
    print(f"  {k}: {round(v, 4) if isinstance(v, float) else v}")

print("\nArtefakty (grafy, tabulky) — najdes ich v MLflow UI:")
print(list(result.artifacts.keys()))

2026/06/30 17:43:04 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


Evaluacny beh: 1dc08a46f6b743d7aa5ee849d33b7b56

Metriky:
  score: 0.4068
  example_count: 133
  mean_absolute_error: 44.114
  mean_squared_error: 3026.203
  root_mean_squared_error: 55.0109
  sum_on_target: 20238.0
  mean_on_target: 152.1654
  r2_score: 0.4068
  max_error: 153.1101
  mean_absolute_percentage_error: 0.3949

Artefakty (grafy, tabulky) — najdes ich v MLflow UI:
[]


## 1.11 Registrácia modelu a nasadenie do staging — Model Registry

**Model Registry** je centrálny katalóg modelov. Každý zaregistrovaný model má **verzie** a voliteľné **aliasy**.

> MLflow 3.x nahradil pevné štádiá (`Staging`, `Production`) **aliasmi** — ľubovoľné reťazce priraďované konkrétnym verziám.

| Krok | Čo robíme |
|------|-----------|
| `mlflow.register_model(uri, name)` | Zaregistruje model z behu, vytvorí verziu |
| `client.set_registered_model_alias(name, alias, version)` | Priradí alias `"staging"` danej verzii |
| `mlflow.pyfunc.load_model("models:/name@staging")` | Načíta model podľa aliasu |

In [121]:
MODEL_NAME = "diabetes-elasticnet"
client = mlflow.tracking.MlflowClient()

# 1) Zaregistruj model z najlepsieho behu do Model Registry
model_uri = f"runs:/{best_run_id}/model"
mv = mlflow.register_model(model_uri, MODEL_NAME)
print(f"Zaregistrovany: {MODEL_NAME} v{mv.version} (run: {best_run_id[:8]}...)")

# 2) Prirad alias "staging" tejto verzii
client.set_registered_model_alias(MODEL_NAME, "staging", mv.version)
print(f"Alias 'staging' -> verzia {mv.version}")

# 3) Overenie: nacitaj model priamo cez alias
staging_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@staging")
preds_staging = staging_model.predict(X_test)
print(f"RMSE staging modelu: {root_mean_squared_error(y_test, preds_staging):.2f}")

Registered model 'diabetes-elasticnet' already exists. Creating a new version of this model...
2026/06/30 17:43:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 60c88e2f4af5450aa1a04aad69918ec5 has no artifacts at artifact path 'model', registering model based on models:/m-3375053374e943eea1cf9082edd2c911 instead


Zaregistrovany: diabetes-elasticnet v4 (run: 60c88e2f...)
Alias 'staging' -> verzia 4
RMSE staging modelu: 55.01


Created version '4' of model 'diabetes-elasticnet'.


## 1.12 Povýšenie modelu do produkcie

Keď je model otestovaný v `staging`, priradíme mu alias `"champion"` — konvencia MLflow 3.x pre produkčný model. Alias `"staging"` zostane zachovaný; jedna verzia môže mať viacero aliasov.

In [122]:
# Ziskaj verziu, ktora ma alias "staging"
staging_version = client.get_model_version_by_alias(MODEL_NAME, "staging").version
print(f"Staging verzia: {staging_version}")

# Prirad alias "champion" (= produkcia) tej istej verzii
client.set_registered_model_alias(MODEL_NAME, "champion", staging_version)
print(f"Alias 'champion' -> verzia {staging_version}")

# Overenie: vypis vsetky aliasy tejto verzie
mv_info = client.get_model_version(MODEL_NAME, staging_version)
print(f"Aliasy verzie {staging_version}: {mv_info.aliases}")

# Nacitaj produkcny model cez alias a skontroluj RMSE
champion_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
preds_champion = champion_model.predict(X_test)
print(f"RMSE champion modelu: {root_mean_squared_error(y_test, preds_champion):.2f}")

Staging verzia: 4
Alias 'champion' -> verzia 4
Aliasy verzie 4: ['champion', 'staging']
RMSE champion modelu: 55.01


## 1.13 Volanie modelu cez REST API — `mlflow models serve`

Pred spustením bunky nižšie spusti mlops server v termináli so staging verziou diabetes-elasticnet modelu:

```bash
mlflow models serve -m "models:/diabetes-elasticnet@staging" --port 5001 --no-conda
```

Endpoint `/invocations` prijíma JSON s kľúčom `dataframe_split`.

In [123]:
import subprocess
import sys
import time
import urllib.request

SERVE_PORT = 5001

server = subprocess.Popen(
    [
        sys.executable, "-m", "mlflow", "models", "serve",
        "-m", f"models:/{MODEL_NAME}@staging",
        "--port", str(SERVE_PORT),
        "--no-conda",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Cakaj kym server naozaj zacne prijimat spojenia (max 30s)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://localhost:{SERVE_PORT}/ping")
        print(f"Server bezi na porte {SERVE_PORT} (PID {server.pid})")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server sa nespustil v limite 30s — pozri logy:")
    print(server.stdout.read().decode())

Server bezi na porte 5001 (PID 5804)


In [124]:
import requests
import json

URL = "http://localhost:5001/invocations"

# Posli 5 vzoriek z testovej sady
sample = X_test.iloc[:5]

payload = {
    "dataframe_split": {
        "columns": sample.columns.tolist(),
        "data": sample.values.tolist(),
    }
}

resp = requests.post(URL, data=json.dumps(payload),
                     headers={"Content-Type": "application/json"})

if resp.status_code == 200:
    predictions = resp.json()["predictions"]
    print(f"{'Skutocnost':>12}  {'Predikcia':>12}")
    print("-" * 27)
    for actual, pred in zip(y_test.iloc[:5], predictions):
        print(f"{actual:>12.1f}  {pred:>12.1f}")
else:
    print(f"Chyba {resp.status_code}: {resp.text}")

  Skutocnost     Predikcia
---------------------------
       321.0         223.0
       215.0         231.9
       127.0         164.0
        64.0         124.0
       175.0         171.8


---
## Čo ďalej — `mlflow.projects`

**MLflow Projects** je štandard na balenie ML kódu do **reprodukovateľných, spustiteľných balíkov**. Rieši problém „u mňa to funguje" — ktokoľvek s prístupom k projektu ho vie spustiť rovnako, bez toho aby ručne inštaloval závislosti alebo hádal správne parametre.

### Štruktúra projektu

Projekt tvorí súbor `MLproject` (YAML) v koreňovom adresári repozitára:

```yaml
name: diabetes-elasticnet

python_env: python_env.yaml   # alebo conda_env / docker_env

entry_points:
  main:
    parameters:
      alpha:    {type: float, default: 0.005}
      l1_ratio: {type: float, default: 0.5}
    command: "python train.py --alpha {alpha} --l1_ratio {l1_ratio}"
```

- **entry points** — čo a ako spustiť; jeden projekt môže mať viac vstupných bodov (tréning, evaluácia, preprocessing...)
- **prostredie** — conda/pip závislosti, alebo Docker image; MLflow prostredie automaticky zreprodukovateľne vytvorí
- **parametre** — s typmi (`float`, `int`, `string`) a predvolenými hodnotami

### Spustenie

Projekt vie ktokoľvek spustiť jediným príkazom — lokálne, z Git URL, alebo vzdialene (Kubernetes, Databricks):

```bash
# lokalne zo sukromneho adresara
mlflow run . -P alpha=0.005 -P l1_ratio=0.5

# priamo z GitHub repozitara (bez klonovania)
mlflow run https://github.com/user/repo -P alpha=0.01
```

Každé spustenie automaticky vytvorí **beh v Trackingu** — parametre, metriky a artefakty sa zalogujú presne tak, ako ste videli v tejto kapitole. Behy sú tak plne sledovateľné a porovnateľné bez ohľadu na to, kto a kde projekt spustil.

### Kedy sa to hodí

| Scenár | Prínos |
|--------|--------|
| Tímová spolupráca | každý spúšťa rovnaké prostredie, žiadne „u mňa to ide" |
| CI/CD pipeline | `mlflow run` ako krok v GitHub Actions / GitLab CI |
| Vzdialené spúšťanie | presun výpočtu na Kubernetes cluster bez zmeny kódu |
| Reprodukovateľnosť | experiment z pred roka spustíš rovnako aj dnes |

Viac: [MLflow Projects docs](https://mlflow.org/docs/latest/projects.html).